In [1]:
import pandas as pd
import geopandas as gpd
import fiona
import glob
from functions import *

fiona.drvsupport.supported_drivers['KML'] = 'rw'

Load data

In [2]:
files = glob.glob("data_origin/geo_data/omi_perimeters/*.kml")

Merge .kml files into a single dataframe

In [3]:
gdf1 = gpd.GeoDataFrame(
    pd.concat([gpd.read_file(f, driver='KML') for f in files], ignore_index=True)
)

In [4]:
gdf = gdf1.copy()

Select relevant columns and clean names

In [5]:
# Select relevant columns
gdf = gdf[['Name', 'CODZONA', 'geometry']]

# Rename columns
gdf = gdf.rename(columns = {
    'Name' : 'mun_name',
    'CODZONA' : 'zone'
})

# Clean mun_name
gdf["mun_name"] = gdf["mun_name"].str.split("-").str[0].str.strip()

# Normalize mun_name and remove "39" (sobstitue for accents and apostrophes)
gdf["mun_name"] = gdf['mun_name'].apply(normalize_name).str.replace(r"39", "", regex = True)

Import ISTAT codes with area intersection

In [6]:
gdf_mun = gpd.read_file("datasets/geo_data/mun_perimeters.gpkg")

c:\Users\HP\Desktop\projects\data_municipalities\.venv_website_housing\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'mun_perimeters.gpkg': 'municipalities' (default), 'mun_perimeters'. Specify layer parameter to avoid this warning.
  result = read_func(


In [7]:
# Create unique zone identifier
gdf = gdf.reset_index(drop=True)
gdf["zone_id"] = gdf.index

gdf = gdf.to_crs(epsg=3857)

gdf_mun = gdf_mun.to_crs(epsg=3857)

# Intersect zones with municipalities
gdf_intersection = gpd.overlay(gdf, gdf_mun[["mun_istat", "geometry"]], how="intersection")

# Calculate intersection area
gdf_intersection["intersection_area"] = gdf_intersection.geometry.area

# Keep only the municipality with the largest overlap per zone
idx = gdf_intersection.groupby("zone_id")["intersection_area"].idxmax()
gdf_best = gdf_intersection.loc[idx, ["zone_id", "mun_istat"]]

# Merge ISTAT code back into gdf 
gdf = gdf.merge(gdf_best, on="zone_id", how="left")

In [11]:
gdf = gdf.drop(columns = 'zone_id')

gdf.to_file("datasets/geo_data/omi_zone_perimeters.gpkg", driver="GPKG", layer="zones")